# AdaBoost AKA Adaptive Boosting

## Decision stump is a decision tree with just one split.

# AdaBoost (Adaptive Boosting)

## 1. Definition

AdaBoost is an ensemble learning algorithm that combines multiple weak learners into a strong classifier by increasing the relative importance of misclassified samples.

## 2. Algorithm

1. Initialize equal weights for all training samples.
2. Train a weak learner using the weighted training data.
3. Calculate its weighted classification error.
4. Calculate the learner's importance (alpha).
5. Update sample weights to emphasize misclassified samples.
6. Repeat for multiple learners.
7. Combine all learners using a weighted vote.

## 3. Important Formulas

### Step 1: Initialize sample weights

For `N` training samples:

$$
w_i = \frac{1}{N}
$$

### Step 2: Calculate weighted error

$$
\epsilon_t =
\sum_{i=1}^{N} w_i^{(t)}
I\left(y_i \neq h_t(x_i)\right)
$$

Where:

- $w_i^{(t)}$: weight of sample $i$ at iteration $t$.
- $y_i$: actual label.
- $h_t(x_i)$: prediction of the current weak learner.
- $I(\cdot)$: equals 1 when the condition is true, otherwise 0.

### Step 3: Calculate learner importance

$$
\alpha_t = \frac{1}{2}
\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)
$$

A lower weighted error gives the learner a higher alpha.

### Step 4: Update sample weights

For labels encoded as $y_i \in \{-1,+1\}$:

$$
w_i^{(t+1)}
=
w_i^{(t)}
\exp\left(-\alpha_t y_i h_t(x_i)\right)
$$

Normalize the weights:

$$
w_i^{(t+1)}
=
\frac{w_i^{(t+1)}}{
\sum_{j=1}^{N} w_j^{(t+1)}
}
$$

- Correct prediction: sample weight decreases.
- Incorrect prediction: sample weight increases.

### Step 5: Final prediction

$$
H(x)=\operatorname{sign}
\left(\sum_{t=1}^{T}\alpha_t h_t(x)\right)
$$

Where:

- $T$: number of weak learners.
- $\alpha_t$: importance of learner $t$.
- $h_t(x)$: prediction of learner $t$.

## 4. Key Points

- AdaBoost is a **boosting algorithm** that trains weak learners sequentially.
- Decision stumps are commonly used as weak learners.
- Misclassified samples receive greater relative importance.
- Learners with lower weighted errors receive higher alpha values.
- The final classification is based on a weighted vote.
- Standard binary AdaBoost requires learners to perform better than random guessing; perfect learners require special handling.

## 5. Advantages and Limitations

**Advantages**
- Simple and effective for many classification tasks.
- Can improve the performance of weak learners.
- Does not require complex neural network architectures.

**Limitations**
- Sensitive to noisy data and outliers.
- Performance depends on the quality of weak learners.
- Sequential training limits parallelization across boosting rounds.

In [ ]:
import numpy as np


class DecisionStump:
    def fit(self, X, y, sample_weights):
        X = np.asarray(X)
        y = np.asarray(y)

        best_error = float("inf")

        for feature in range(X.shape[1]):
            values = np.unique(X[:, feature])
            thresholds = (values[:-1] + values[1:]) / 2

            for threshold in thresholds:
                for polarity in [1, -1]:
                    predictions = np.ones(len(y), dtype=int)

                    if polarity == 1:
                        predictions[X[:, feature] <= threshold] = -1
                    else:
                        predictions[X[:, feature] > threshold] = -1

                    error = np.sum(
                        sample_weights * (predictions != y)
                    )

                    if error < best_error:
                        best_error = error
                        self.feature = feature
                        self.threshold = threshold
                        self.polarity = polarity

        self.error = best_error
        return self

    def predict(self, X):
        X = np.asarray(X)
        predictions = np.ones(len(X), dtype=int)

        if self.polarity == 1:
            predictions[X[:, self.feature] <= self.threshold] = -1
        else:
            predictions[X[:, self.feature] > self.threshold] = -1

        return predictions


class AdaBoost:
    def __init__(self, n_estimators=10):
        self.n_estimators = n_estimators
        self.models = []
        self.alphas = []

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        # AdaBoost uses labels -1 and +1
        y = np.where(y == 0, -1, 1)

        n_samples = len(y)
        sample_weights = np.ones(n_samples) / n_samples

        self.models = []
        self.alphas = []

        for _ in range(self.n_estimators):
            stump = DecisionStump()
            stump.fit(X, y, sample_weights)

            predictions = stump.predict(X)
            
            error = np.sum(
                sample_weights * (predictions != y)
            )

            # Stop if the learner is no better than random guessing
            if error >= 0.5:
                break

            # Avoid division by zero for a perfect learner
            error = max(error, 1e-12)

            alpha = 0.5 * np.log((1 - error) / error)

            # Update sample weights
            sample_weights *= np.exp(
                -alpha * y * predictions
            )
            
            sample_weights /= sample_weights.sum()

            self.models.append(stump)
            self.alphas.append(alpha)

            if error <= 1e-12:
                break

        return self

    def predict(self, X):
        if not self.models:
            raise ValueError("No weak learners were trained.")

        predictions = np.array([
            alpha * model.predict(X)
            for model, alpha in zip(self.models, self.alphas)
        ])

        final_scores = predictions.sum(axis=0)
        return np.where(final_scores >= 0, 1, -1)


# Example dataset

X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
# X = np.array([[1, 10], [2, 9], [3, 8], [4, 7], [5, 6], [6, 5], [7, 4], [8, 3], [9, 2], [10, 1]])

y = np.array([0, 0, 0, 1, 0, 1, 1, 1, 1, 1])

# Train AdaBoost
model = AdaBoost(n_estimators=10)
model.fit(X, y)

# Predict
predictions = model.predict(X)

# Convert predictions back to 0 and 1
predictions = np.where(predictions == -1, 0, 1)

accuracy = np.mean(predictions == y)

print("Number of learners:", len(model.models))
print("Learner weights (alphas):", model.alphas)
print("Actual labels:   ", y)
print("Predicted labels:", predictions)
print(f"Training accuracy: {accuracy:.2%}")

Number of learners: 10
Learner weights (alphas): [np.float64(1.0986122886681098), np.float64(1.416606672028108), np.float64(0.5893274981708231), np.float64(0.7819877691786715), np.float64(0.7004465802705216), np.float64(0.7302011666368062), np.float64(0.7186484577861316), np.float64(0.7230330522099039), np.float64(0.7213542251999909), np.float64(0.7219948850507154)]
Actual labels:    [0 0 0 1 0 1 1 1 1 1]
Predicted labels: [0 0 0 1 0 1 1 1 1 1]
Training accuracy: 100.00%
